# 000. 원격 갱신 (Colab) — 태블릿에서 refresh_all 돌리기

밖에서 태블릿만 있을 때 **데이터 갱신 + 실서비스 배포**까지 하는 노트북이다.
1~6번 노트북(크롤링·검증·수비이닝·경기일정)은 원래 Colab 이라 그대로 돌리면 되고,
막히는 건 로컬 PC 에서만 돌던 `scripts/refresh_all.py` 하나였다. 그걸 여기서 돌린다.

## 나가기 전에 딱 한 번 해둘 것 — Colab 시크릿 2개

왼쪽 **🔑(보안 비밀)** 아이콘 → `새 보안 비밀 추가` 로 아래 두 개를 넣고,
**둘 다 '노트북 액세스' 토글을 켠다.** 계정에 저장되므로 태블릿에서도 그대로 쓰인다.

| 이름 | 값 |
|---|---|
| `GH_TOKEN` | GitHub Personal Access Token (repo 권한). github.com → Settings → Developer settings → Personal access tokens |
| `GCP_SA_TOML` | PC 의 `졸업작품/secrets/secrets.toml` **파일 내용 전체**를 그대로 복사·붙여넣기 |

`secrets.toml` 은 gitignore 되어 GitHub 에 없다. 그래서 시크릿으로 넣어야 한다.
**이 두 개는 PC 에서 미리 넣어둬야 한다** — 나가서는 파일을 못 여니까.

## 돌리는 순서

1. 1번 크롤링 노트북 → 2·3번 검증 → 4번 → 5·6번 (평소대로)
2. 이 노트북의 아래 두 셀 실행
3. 끝나면 GitHub Actions 가 실서비스에 배포한다 (1~2분)

In [ ]:
# ── 준비: 저장소 내려받기 + 키 복원 + 패키지 설치 ──────────────────
import shutil, subprocess
from pathlib import Path
from google.colab import userdata

GH_TOKEN = userdata.get("GH_TOKEN")
SA_TOML = userdata.get("GCP_SA_TOML")
assert GH_TOKEN and SA_TOML, "🔑 시크릿 GH_TOKEN / GCP_SA_TOML 을 먼저 넣으세요"

BASE = Path("/content/hs")
PIPE, WEB = BASE / "pipeline", BASE / "web"


def run(cmd, cwd=None):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    # clone 주소에 토큰이 들어 있다. 그대로 찍으면 출력째 커밋돼 토큰이 샌다.
    print("$", shown.replace(GH_TOKEN, "***"))
    p = subprocess.run(cmd, shell=isinstance(cmd, str), text=True, cwd=cwd)
    if p.returncode:
        raise SystemExit(f"실패(코드 {p.returncode}): {cmd}")


# Colab VM 은 일회용이라 매번 새로 받는다(중간 상태가 남아 헷갈릴 일이 없다).
if BASE.exists():
    shutil.rmtree(BASE)
BASE.mkdir(parents=True)
auth = f"https://{GH_TOKEN}@github.com/Ryong1001"
run(["git", "clone", "--quiet", f"{auth}/highschool-stats-pipeline.git", str(PIPE)])
run(["git", "clone", "--quiet", f"{auth}/highschool-stats-web.git", str(WEB)])

# 서비스 계정 키는 저장소에 없으므로 시크릿에서 복원한다.
(PIPE / "secrets").mkdir(exist_ok=True)
(PIPE / "secrets" / "secrets.toml").write_text(SA_TOML, encoding="utf-8")

# 커밋 author 가 소유 계정이 아니면 Vercel(Hobby) 이 배포를 Blocked 한다.
for repo in (PIPE, WEB):
    run(["git", "-C", str(repo), "config", "user.name", "Ryong1001"])
    run(["git", "-C", str(repo), "config", "user.email", "owner@example.com"])

run(f"pip install -q -r {PIPE}/requirements.txt")
print()
print("준비 완료. web 브랜치:", subprocess.run(
    ["git", "-C", str(WEB), "branch", "--show-current"],
    capture_output=True, text=True).stdout.strip(), "(master = 실서비스)")
print("⚠ pip 이 pandas/numpy 를 새로 깔았다면 [런타임] → [세션 다시 시작] 후 이 셀부터 다시 실행")

## (선택) 노트북을 Drive 에 최신으로 맞추기

PC 에서 고친 노트북은 GitHub 에는 올라가 있지만 Drive `_검수 코드` 폴더에는
자동으로 반영되지 않는다. 태블릿에서 Colab 이 여는 건 Drive 사본이므로,
**옛 버전을 돌리지 않으려면 세션 시작할 때 아래 셀을 한 번 돌린다.**

(PC 쪽 `scripts/sync_notebooks.py` 는 서비스 계정 용량 제한으로 못 쓴다.
 아래처럼 Colab 에서 내 계정으로 하는 게 유일하게 되는 방법이다.)

In [ ]:
# ── (선택) 노트북 000·1~7 을 Drive '_검수 코드' 폴더에 최신으로 맞추기 ────────
# PC 에서 서비스 계정으로 올리려 했더니 막혔다 — 서비스 계정은 Drive 저장 용량이
# 없어서 새 파일을 만들 수 없다(storageQuotaExceeded). 폴더 공유 문제가 아니다.
# 여기서는 내 계정으로 Drive 를 마운트하므로 그 제약이 없다. GitHub 사본을 복사한다.
# 위 준비 셀이 먼저 실행돼 있어야 한다(저장소가 /content/hs/pipeline 에 받아져 있음).
import re
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DEST = Path("/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)/_검수 코드")
DEST.mkdir(parents=True, exist_ok=True)

# 원격 갱신 노트북은 목록 맨 앞에 오도록 000. 으로 바꿨다(옛 이름 7.) — 둘 다 잡고, 옛 사본은 지운다.
PAT = re.compile(r"^(000|[1-7](-\d)?)\. .*\.ipynb$")
OLD = DEST / "7. 원격 갱신 (Colab).ipynb"
if OLD.exists():
    OLD.unlink()
    print("✕ 옛 사본 삭제:", OLD.name)
n = 0
for src in sorted(Path("/content/hs/pipeline").glob("*.ipynb")):
    if PAT.match(src.name):
        shutil.copy2(src, DEST / src.name)
        print("↑", src.name)
        n += 1
print()
print(f"{n}개를 {DEST} 에 덮어썼습니다")

## 갱신 실행

아래 셀이 export 를 전부 다시 굽고 → web 저장소에 커밋 → 푸시한다.
푸시되면 web 의 **Deploy** 워크플로가 이어받아 실서비스에 배포한다.

**10~30분** 걸린다. 도중에 태블릿 화면이 꺼지면 세션이 끊길 수 있으니
화면을 켜두거나 가끔 탭을 확인하는 게 안전하다.

In [ ]:
# ── 갱신 실행 (10~30분) ──────────────────────────────────────────
%env HS_WEB_ROOT=/content/hs/web
%env PYTHONIOENCODING=utf-8
!cd /content/hs/pipeline && python scripts/refresh_all.py 2026

## 잘 안 될 때

| 증상 | 대처 |
|---|---|
| `시크릿 ... 먼저 넣으세요` | 🔑 에서 이름 철자 확인, **'노트북 액세스' 토글**이 켜져 있는지 확인 |
| `web 저장소를 찾지 못했습니다` | 준비 셀이 실패한 것. 위 셀부터 다시 |
| `ModuleNotFoundError` / 버전 충돌 | [런타임] → [세션 다시 시작] 후 준비 셀부터 다시 |
| `index.lock` | Colab 에서는 안 나지만, 나면 `!rm -f /content/hs/web/.git/index.lock` |
| author 경고로 중단 | 준비 셀의 git config 가 안 먹은 것. 다시 실행 |
| 푸시했는데 사이트가 그대로 | GitHub → highschool-stats-web → **Actions** 탭에서 Deploy 실행 상태 확인 |

### 데이터를 안 바꾸고 재배포만 하고 싶으면
이 노트북 없이도 된다 — GitHub → `highschool-stats-web` → **Actions** → **Deploy**
→ **Run workflow** → 브랜치 `master`.

### 특정 export 만 다시 굽고 싶으면
`!cd /content/hs/pipeline && python scripts/refresh_all.py 2026 --only rankings,teams`